In [ ]:
#Imports
!pip install requests cryptography

import time
import base64
import requests
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives.serialization import load_pem_private_key
import pandas as pd

In [ ]:
#Initialize API information and basic functions
KEY_ID = "db85ac8b-dfa4-4fc1-b9cf-641ea46b448a"

with open("KalshiKey.txt", "r") as f:
    PRIVATE_KEY_STRING = f.read()

PRIVATE_KEY_STRING = PRIVATE_KEY_STRING.encode("utf-8")

BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"

private_key = load_pem_private_key(
    PRIVATE_KEY_STRING,
    password=None,
)

def sign_request(method, path, timestamp):
    message = f"{timestamp}{method}{path}".encode()

    signature = private_key.sign(
        message,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH,
        ),
        hashes.SHA256(),
    )

    return base64.b64encode(signature).decode()

def make_request(method, path, params=None):
    timestamp = str(int(time.time() * 1000))
    signature = sign_request(method, path, timestamp)
    
    headers = {
        "KALSHI-ACCESS-KEY": KEY_ID,
        "KALSHI-ACCESS-SIGNATURE": signature,
        "KALSHI-ACCESS-TIMESTAMP": timestamp,
    }
    
    url = BASE_URL + path
    response = requests.request(method, url, headers=headers, params=params)
    
    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        return None
    try:
        return response.json()
    except Exception:
        print("Non-JSON response:", response.text)
        return None

In [ ]:
#Preview market information
markets = make_request("GET", "/historical/markets", params={"limit": 1})

target_start = 1000
target_end = 1200
page_size = 100

all_markets = []
cursor = None
count = 0

while True:
    params = {"limit": page_size}
    if cursor:
        params["cursor"] = cursor
    
    data = make_request("GET", "/historical/markets", params=params)
    time.sleep(0.01)
    if not data or "markets" not in data:
        break
    
    markets = data["markets"]
    count += len(markets)
    
    for m in markets:
        if len(all_markets) < target_end - target_start:
            if count - len(markets) + markets.index(m) >= target_start:
                all_markets.append(m)
    
    cursor = data.get("cursor")
    if not cursor or len(all_markets) >= (target_end - target_start):
        break

print(f"Retrieved {len(all_markets)} markets (1000th–1200th)")
for i in all_markets:
    print(i)

In [ ]:
#Initialize data extraction methods
def get_all_markets():
    all_markets = []
    cursor = None

    while True:
        params = {"limit": 1000}
        if cursor:
            params["cursor"] = cursor

        data = make_request("GET", "/historical/markets", params=params)
        
        if not data or "markets" not in data:
            break
        
        all_markets.extend(data["markets"])
        cursor = data.get("cursor")

        if not cursor:
            break

    print(f"Total markets fetched: {len(all_markets)}")
    return all_markets

def extract_market_info(markets):
    rows = []
    
    for m in markets:
        rows.append({
            "title": m.get("title"),
            "close_time": m.get("close_time"),
            "last_price_dollars": m.get("last_price_dollars"),
            "result": m.get("result"),
        })
    
    return pd.DataFrame(rows)

def build_dataset():
    markets = get_all_markets()
    df = extract_market_info(markets)
    return df

In [ ]:
#Extract data
df = build_dataset()
df.to_csv("output.csv", index=False)

Total markets fetched: 1012877
